# Data Quality

This notebook logs structural quality checks for the raw dataset and records any issues for review. No values are fixed here.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path('data/raw/Liberty_Heritage_Data.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/raw/Liberty_Heritage_Data.csv')

df = pd.read_csv(DATA_PATH)

print(f'Loaded {DATA_PATH} with shape {df.shape}')
print(df.head().to_string(index=False))

Loaded ../data/raw/Liberty_Heritage_Data.csv with shape (8000, 25)
ApplicationID  Age Gender MaritalStatus  Dependents EducationLevel State       ResidenceType  YearsAtCurrentResidence      EmploymentType  Employed  EmploymentLengthYears  AnnualIncome  IncomeVerified  CreditScore  CreditHistoryMonths  ExistingLoanAccounts  ExistingCreditCards  TotalMonthlyDebtPayment  DebtToIncomeRatio  RevolvingUtilization  PriorDefault  BankruptcyLast7Years  EnquiriesLast6Months Approval
  LHB-2403641   60 Female      Divorced           4     Bachelor's    GA                Rent                      2.0 Salaried-Government         1                    2.0         100.2               1          513                  369                     4                    3                   2271.0              0.272                 0.484             1                     0                     3       No
  LHB-2405086   28   Male       Married           1      Doctorate    KY Own (with mortgage)                   

## Structural Checks

The following checks report the dataset size, duplicate rows, missing values, and selected anomaly rules.

In [2]:
row_count = int(len(df))
duplicate_rows = int(df.duplicated().sum())

missing_counts = df.isna().sum()
missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': (missing_counts / row_count * 100).round(2),
}).sort_values(['missing_count', 'missing_pct'], ascending=False)

age_anomaly_count = int((df['Age'] < 18).sum())
credit_score_anomaly_count = int((~df['CreditScore'].between(300, 850, inclusive='both')).sum())
annual_income_anomaly_count = int((df['AnnualIncome'] <= 0).sum())

quality_findings = {
    'row_count': row_count,
    'duplicate_rows': duplicate_rows,
    'missing_summary': missing_summary,
    'age_less_than_18_count': age_anomaly_count,
    'credit_score_outside_300_850_count': credit_score_anomaly_count,
    'annual_income_zero_or_below_count': annual_income_anomaly_count,
}

print(f'Row count: {row_count}')
print(f'Duplicate rows: {duplicate_rows}')
print('\nMissing values by column:')
print(missing_summary[missing_summary['missing_count'] > 0].to_string())
print('\nAnomaly checks:')
print(f"Age < 18: {age_anomaly_count}")
print(f"CreditScore outside 300-850: {credit_score_anomaly_count}")
print(f"AnnualIncome <= 0: {annual_income_anomaly_count}")
print('\nStructured findings:')
print(quality_findings)

Row count: 8000
Duplicate rows: 0

Missing values by column:
                         missing_count  missing_pct
EmploymentLengthYears              146         1.82
AnnualIncome                       139         1.74
YearsAtCurrentResidence             96         1.20
RevolvingUtilization                85         1.06
DebtToIncomeRatio                   75         0.94

Anomaly checks:
Age < 18: 0
CreditScore outside 300-850: 0
AnnualIncome <= 0: 0

Structured findings:
{'row_count': 8000, 'duplicate_rows': 0, 'missing_summary':                          missing_count  missing_pct
EmploymentLengthYears              146         1.82
AnnualIncome                       139         1.74
YearsAtCurrentResidence             96         1.20
RevolvingUtilization                85         1.06
DebtToIncomeRatio                   75         0.94
ApplicationID                        0         0.00
Age                                  0         0.00
Gender                               0         0

## Prior Default vs Bankruptcy

In principle, a recent bankruptcy should usually correspond to a prior default flag, so this cross-tab shows all four combinations for review.

In [3]:
prior_default_bankruptcy_crosstab = pd.crosstab(
    df['PriorDefault'],
    df['BankruptcyLast7Years'],
    dropna=False,
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

print('PriorDefault vs BankruptcyLast7Years')
print(prior_default_bankruptcy_crosstab.to_string())

PriorDefault vs BankruptcyLast7Years
BankruptcyLast7Years     0    1
PriorDefault                   
0                     7155  153
1                      673   19


## Review Notes

- Log any columns with non-zero missing values for downstream review.
- Confirm whether the zero-anomaly checks should be treated as hard validation rules or soft business rules.
- Keep this notebook as a record of the raw-state audit before any cleaning or transformation.